In [12]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN
import kmapper as km

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
FILE_NAME = Path('Data') / 'Ingesta_AF_clean.csv'
print(FILE_NAME.resolve())
df = pd.read_csv(FILE_NAME)

print("Shape del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()


C:\Users\Emiliano\Desktop\Python\TopoProj\Data\Ingesta_AF_clean.csv
Shape del dataset: (1170, 73)

Primeras filas:


,Cod,edad,region_chile,neduc,paridad,fnacimiento,Sexo,pnacer,Edad_gestacion ultimo hijo,sitgest,...,pnacer_raw,eg_raw,uf_af,n_panes,pnacer_num,eg_num,n_panes_num,prematuro,eg_cat,uso_supl
0,1,26.0,1,4,1,2015-01-15 00:00:00,2,3040,39,9,...,3040.0,39.0,1000.0,2.0,3040.0,39.0,2.0,0.0,39–40 (normal),Usó suplemento
1,2,36.0,1,6,2,2012-03-29 00:00:00,2,3875,39,5,...,3875.0,39.0,1000.0,3.0,3875.0,39.0,3.0,0.0,39–40 (normal),Usó suplemento
2,3,22.0,1,5,1,2011-11-22 00:00:00,1,2665,38,0,...,2665.0,38.0,NaN,2.0,2665.0,38.0,2.0,0.0,37–38 (temprano),No usó suplemento
3,4,35.0,1,3,2,2007-02-14 00:00:00,1,3450,37,0,...,3450.0,37.0,1000.0,2.0,3450.0,37.0,2.0,0.0,37–38 (temprano),Usó suplemento
4,5,33.0,1,5,0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,3.0,NaN,NaN,3.0,NaN,NaN,No usó suplemento


In [13]:
# VARIABLES Y FEATURES DERIVADAS

selected_cols = [
    "uf-af",
    "N° PANES",

    "Marraqueta",
    "Hallulla",
    "Pan molde integral",

    "Antes del embarazo",
    "Durante todo el embarazo",

    "1-3 meses",
    "4-6 meses",
    "7-9 meses",

    "NO consumio",
    "pnacer",
    "Edad_gestacion ultimo hijo",
    "Preeclamsia",
    "Diabetes gestacional",
    "Embarazo multiple",
    "edad",
    "neduc"
]

data = df[selected_cols].copy()

# Conversión robusta a numérico
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Nuevas variables clínicas / derivadas
data["prematuro"] = (data["Edad_gestacion ultimo hijo"] < 37).astype(int)              # prematuro < 37 semanas
data["bajo_peso"]  = (data["pnacer"] < 2500).astype(int)                               # bajo peso < 2500 g

# Score de riesgo obstétrico (suma ponderada simple: igual peso para cada factor)
data["riesgo_obstetrico"] = (
    data["Preeclamsia"].fillna(0)
    + data["Diabetes gestacional"].fillna(0)
    + data["Embarazo multiple"].fillna(0)
    + data["bajo_peso"].fillna(0)
    + data["prematuro"].fillna(0)
)

# Score de fortificación por pan: se define como N° PANES * 160 µg (estándar de fortificación)
data["score_pan"] = data["N° PANES"].fillna(0) * 160

# Score global de exposición a ácido fólico: suplementación directa (uf-af) + fortificación por pan
data["AF_exposure"] = (
    data["uf-af"].fillna(0)
    + data["score_pan"].fillna(0)
)

print("\nData con features derivadas:")
data.head()


Data con features derivadas:


,uf-af,N° PANES,Marraqueta,Hallulla,Pan molde integral,Antes del embarazo,Durante todo el embarazo,1-3 meses,4-6 meses,7-9 meses,...,Preeclamsia,Diabetes gestacional,Embarazo multiple,edad,neduc,prematuro,bajo_peso,riesgo_obstetrico,score_pan,AF_exposure
0,1000.0,2.0,0,1,0,0,0,0,0,0,...,0,0.0,0.0,26.0,4,0,0,0.0,320.0,1320.0
1,1000.0,3.0,1,0,0,0,0,1,0,0,...,0,1.0,0.0,36.0,6,0,0,1.0,480.0,1480.0
2,NaN,2.0,1,0,0,0,0,0,0,0,...,0,0.0,0.0,22.0,5,0,0,0.0,320.0,320.0
3,1000.0,2.0,1,0,0,0,0,1,0,0,...,0,0.0,0.0,35.0,3,0,0,0.0,320.0,1320.0
4,NaN,3.0,1,0,0,0,0,0,0,0,...,0,0.0,0.0,33.0,5,0,0,0.0,480.0,480.0


In [14]:
data["riesgo_obstetrico"].describe()

count    1170.000000
mean        0.280342
std         0.587908
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         3.000000
Name: riesgo_obstetrico, dtype: float64

### Interpretación del Grafo Mapper

El análisis Mapper, parte de la Topología de Datos Aplicada (TDA), nos permite visualizar la forma y la estructura subyacente de datos complejos de alta dimensión de una manera geométrica. A continuación, se presenta una interpretación completa de este grafo, considerando los parámetros utilizados y la variable de colorización.

#### 1. ¿Qué es el Grafo Mapper y Cómo se Construye?

El algoritmo Mapper construye un grafo (una red de nodos y enlaces) que representa la topología de los datos. Lo hace en tres pasos principales:

1.  **Función Filtro (Lens Function)**: Reduce la dimensionalidad de los datos a un espacio de menor dimensión (generalmente 1, 2 o 3 dimensiones) manteniendo las relaciones topológicas clave. En este caso, se utilizó **t-SNE** (`TSNE(n_components=2, perplexity=35, random_state=42)`) para proyectar `X_scaled` (los datos imputados y escalados con las variables `tda_features`) a un espacio bidimensional. La perplejidad de 35 influye en cómo t-SNE balancea la atención en estructuras locales y globales.
2.  **Cubre (Cover)**: Se define una "cubierta" sobre el espacio de la función filtro. Esto implica dividir el espacio de la lente en varios intervalos superpuestos. Para este grafo, se usó `km.Cover(n_cubes=12, perc_overlap=0.45)`, lo que significa que el espacio bidimensional de la lente se dividió en 12 intervalos por cada dimensión (total 144 regiones), con un solapamiento del 45% entre ellos. Este solapamiento es crucial para capturar la conectividad de los datos.
3.  **Agrupamiento Local (Local Clustering)**: Dentro de cada "cubo" o región de la cubierta, se agrupan los puntos de datos que caen dentro de él utilizando un algoritmo de clustering local. Aquí se aplicó **DBSCAN** (`DBSCAN(eps=0.9, min_samples=6)`). `eps` define el radio máximo de distancia para que dos puntos sean considerados vecinos, y `min_samples` es el número mínimo de puntos para formar un clúster denso. Si un cubo no tiene suficientes puntos para formar un clúster, no se crea un nodo.

*   **Nodos**: Cada nodo en el grafo representa uno o más clústeres de datos locales dentro de una región de la cubierta. Los puntos de datos dentro de un nodo son topológicamente similares según la función filtro y el criterio de clustering.
*   **Enlaces**: Un enlace entre dos nodos indica que comparten al menos un punto de datos o que sus regiones cubiertas se superponen y contienen puntos similares.

#### 2. Características del Grafo Generado

El grafo construido tiene las siguientes características:

*   **Nodos**: `61`
*   **Enlaces**: `48`

Un número moderado de nodos y enlaces sugiere que se ha logrado una segmentación de los datos que revela una estructura sin ser excesivamente granular o demasiado simplificada. La conectividad (48 enlaces entre 61 nodos) indica que hay relaciones y transiciones entre los diferentes grupos identificados.

#### 3. Interpretación de la Colorización ('Exposición Total AF')

La visualización interactiva del grafo colorea los nodos basándose en la variable `AF_exposure` (Exposición Total a Ácido Fólico). Esta variable es un **score global de exposición a ácido fólico** que combina la suplementación directa (`uf-af`) y la fortificación por pan (`score_pan`).

*   **Gradiente de color**: Los nodos con colores más intensos (o un color específico, dependiendo de la paleta) representarán clústeres de individuos con una mayor exposición al ácido fólico, mientras que los nodos con colores más tenues o diferentes tonalidades representarán menor exposición. (En la visualización, el color "Exposición Total AF" permite ver cómo se distribuye esta métrica a través de la topología de los datos).

#### 4. Hallazgos y Patrones Potenciales

Al observar la visualización interactiva de este grafo, se buscarían los siguientes patrones y hallazgos:

*   **Subgrupos Distintos**: La presencia de varias "ramas" o "componentes conectadas" en el grafo puede indicar la existencia de subgrupos distintos de mujeres embarazadas. Por ejemplo, podría haber un grupo de mujeres con alta exposición al AF y otro con baja, y posiblemente grupos intermedios.
*   **Heterogeneidad dentro de los Grupos**: Si un nodo es grande o si varios nodos cercanos tienen colores similares, pero están conectados a otros nodos con colores diferentes, esto podría indicar que hay una continuidad o transición entre diferentes niveles de exposición al AF, o que hay heterogeneidad en otras características (como `pnacer`, `prematuro`, `riesgo_obstetrico`, `edad`, `neduc`) dentro de un rango similar de exposición al AF.
*   **Nodos Críticos / Puentes**: Los nodos que sirven como "puentes" que conectan diferentes ramas del grafo son interesantes. Podrían representar poblaciones de transición o puntos de inflexión en las características de las madres.
*   **Nodos Aislados o Pequeños**: Nodos que no tienen muchos enlaces o que son muy pequeños podrían representar subgrupos minoritarios o anómalos dentro de la población de estudio. Si estos nodos aislados presentan valores extremos de `AF_exposure` (muy altos o muy bajos), podrían ser poblaciones de especial interés.
*   **Relación con Desenlaces Neonatales y Riesgos Maternos**: Aunque la colorización es por `AF_exposure`, la construcción del grafo considera todas las `tda_features`. Por lo tanto, los clústeres (nodos) identificados por su patrón de `AF_exposure` están implícitamente relacionados con las otras variables en `tda_features` (peso al nacer, edad gestacional, prematuridad, bajo peso, riesgo obstétrico, edad y nivel educacional). Por ejemplo, un clúster con baja `AF_exposure` podría correlacionarse con una mayor incidencia de prematuridad o bajo peso al nacer, si los datos lo apoyan, y viceversa.

#### 5. Conclusión

El grafo Mapper es una herramienta exploratoria potente para revelar la estructura oculta y las relaciones no lineales en el dataset. La visualización con `AF_exposure` como variable de colorización es clave para identificar cómo los diferentes patrones de exposición al ácido fólico se agrupan y se conectan en la población de estudio, y cómo estos grupos se diferencian en términos de las otras características clínicas y sociodemográficas incluidas en el análisis.

In [15]:
# VARIABLES PARA EL ESPACIO MÉTRICO (Mapper)

tda_features = [
    # Exposición AF
    "AF_exposure",
    "uf-af",
    "score_pan",

    # Timing de suplementación
    "Antes del embarazo",
    "Durante todo el embarazo",
    "1-3 meses",
    "4-6 meses",
    "7-9 meses",

    # Características maternas
    "edad",
    "neduc"
]

X = data[tda_features].copy()

In [16]:
# IMPUTACIÓN DE VALORES NULOS Y ESCALAMIENTO

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Forma de X_scaled:", X_scaled.shape)

Forma de X_scaled: (1170, 10)


In [17]:
# CREACIÓN DEL MAPPER Y LENSA FILTRO (TSNE)

from sklearn.decomposition import PCA


mapper = km.KeplerMapper(verbose=1)

lens = mapper.project(
    X_scaled,
    projection=PCA(
        n_components=2,
        random_state=42
    )
)

KeplerMapper(verbose=1)
..Projecting on data shaped (1170, 10)

..Projecting data using: 
	PCA(n_components=2, random_state=42)


..Scaling with: MinMaxScaler()



In [18]:
# CONFIGURACIÓN DE LA CUBIERTA (COVER)

cover = km.Cover(
    n_cubes=15,           # número de intervalos
    perc_overlap=0.50     # porcentaje de solapamiento
)

In [19]:
# CLUSTERING LOCAL (DBSCAN)

from sklearn.cluster import AgglomerativeClustering


clusterer = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward'
)

In [20]:
# CONSTRUCCIÓN DEL GRAFO MAPPER

graph = mapper.map(
    lens,
    X_scaled,
    cover=cover,
    clusterer=clusterer
)

print("Nodos:", len(graph["nodes"]))
print("Links:", len(graph["links"]))

Mapping on data shaped (1170, 10) using lens shaped (1170, 2)

Creating 225 hypercubes.

Created 1507 edges and 564 nodes in 0:00:00.127703.
Nodos: 564
Links: 491


In [ ]:
# VISUALIZACIÓN INTERACTIVA – COLOR FUNCTION

# Elije una de las tres opciones (descomenta una y comenta las otras)

# OPCIÓN A — exposición total a ácido fólico
color_values = X_imputed["AF_exposure"]
color_name   = "Exposición Total AF"

# OPCIÓN B — prematuridad
#color_values = X_imputed["prematuro"]
#color_name   = "Prematuridad"

# OPCIÓN C — peso al nacer
# color_values = X_imputed["pnacer"]
# color_name   = "Peso al nacer"


html = mapper.visualize(
    graph,
    path_html="mapper_af_embarazo_semifinal.html",

    title="""
    TDA Mapper — Segmentación de Mujeres Embarazadas
    según Exposición a Ácido Fólico y Riesgo Neonatal
    """,

    color_values=color_values,

    color_function_name=color_name
)

Wrote visualization to: mapper_af_embarazo.html


In [22]:
# EXTRAER COMPONENTES CONECTADOS DEL GRAFO MAPPER (Union-Find)
from collections import defaultdict
from pathlib import Path
import pickle

print('=' * 70)
print('IDENTIFICANDO COMPONENTES CONECTADOS DEL GRAFO MAPPER')
print('=' * 70)

# Inspeccionar estructura de graph['links']
print(f'\nEstructura de graph["links"]:')
print(f'  Tipo: {type(graph["links"])}')
print(f'  Número de elementos: {len(graph["links"])}')
if len(graph["links"]) > 0:
    print(f'  Primer elemento: {graph["links"][0]}')
    print(f'  Tipo del primer elemento: {type(graph["links"][0])}')

# 1) Union-Find para identificar componentes conectados desde graph["links"]
parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

# Inicializar todos los nodos
for node_id in graph['nodes'].keys():
    find(node_id)

# Unir nodos conectados por links (manejar ambos formatos: tupla o diccionario)
print(f'\nProcesando links...')
for link_item in graph['links']:
    if isinstance(link_item, (list, tuple)) and len(link_item) == 2:
        # Formato: [node_a, node_b] o (node_a, node_b)
        node_a, node_b = link_item
    elif isinstance(link_item, dict):
        # Formato: {"source": node_a, "target": node_b} o similar
        if 'source' in link_item and 'target' in link_item:
            node_a, node_b = link_item['source'], link_item['target']
        else:
            # Intentar extraer los dos primeros valores
            vals = list(link_item.values())
            if len(vals) >= 2:
                node_a, node_b = vals[0], vals[1]
            else:
                print(f'  ⚠ Formato de link no reconocido: {link_item}')
                continue
    else:
        print(f'  ⚠ Formato de link no reconocido: {link_item} ({type(link_item)})')
        continue
    
    union(node_a, node_b)

# Agrupar nodos por componente conectada
components = defaultdict(list)
for node_id in graph['nodes'].keys():
    comp_id = find(node_id)
    components[comp_id].append(node_id)

print(f'\nComponentes conectados encontrados: {len(components)}')
for i, (comp_id, nodes) in enumerate(sorted(components.items())):
    n_obs = sum(len(graph['nodes'][n]) for n in nodes)
    print(f'  Componente {i}: {len(nodes)} nodos → {n_obs} observaciones')

# 2) Mapear observaciones a componentes conectadas
point_to_component = {}
for comp_id, nodes_in_comp in components.items():
    for node_id in nodes_in_comp:
        for point_idx in graph['nodes'][node_id]:
            try:
                point_idx_int = int(point_idx)
            except:
                point_idx_int = point_idx
            point_to_component[point_idx_int] = comp_id

# 3) Asignar componente a cada observación (usando índice posicional en data)
assign = pd.Series(index=data.index, dtype="object")
for pos, comp_id in point_to_component.items():
    if pos < len(data):
        orig_idx = data.index[pos]
        assign.loc[orig_idx] = comp_id

data["cluster_mapper"] = assign

# 4) Filtrar componentes por tamaño mínimo (n >= 20)
comp_counts = data['cluster_mapper'].value_counts()
valid_comps = comp_counts[comp_counts >= 20].index.tolist()

print(f'\nFiltro: componentes con n >= 20')
print(f'  Componentes válidos: {len(valid_comps)} de {len(components)}')
for comp in sorted(valid_comps):
    n = comp_counts[comp]
    print(f'    Componente {comp}: n={int(n)}')

# Crear mapeo: componentes_válidas → nuevos IDs (0, 1, 2, ...)
comp_remapping = {old_comp: new_id for new_id, old_comp in enumerate(sorted(valid_comps))}

# Reasignar solo componentes válidos (descartar el resto como NaN)
data["cluster_mapper"] = data["cluster_mapper"].map(comp_remapping)

print(f'\nResumen final:')
print(f'  Observaciones con componente válido: {data["cluster_mapper"].notna().sum()} / {len(data)}')
print(f'  Número de componentes en análisis: {data["cluster_mapper"].max() + 1 if data["cluster_mapper"].notna().any() else 0}')

# 5) Guardar grafo completo en pickle (para HomologiaPersistente)
graph_path = Path('mapper_graph.pkl')
try:
    with open(graph_path, 'wb') as f:
        pickle.dump(graph, f)
    print(f'\n✓ Grafo Mapper guardado en {graph_path}')
except Exception as e:
    print(f'\n✗ Error guardando grafo: {e}')

# 6) Exportar CSV con componentes conectados filtrados
out_df = pd.DataFrame({
    "orig_index": data.index,
    "cluster_mapper": data["cluster_mapper"].values
})

save_paths = [Path('Data') / 'mapper_clusters.csv', Path('mapper_clusters.csv')]
for p in save_paths:
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        out_df.to_csv(p, index=False)
        print(f'✓ CSV guardado: {p}')
    except Exception as e:
        print(f'✗ Error guardando CSV en {p}: {e}')

print('=' * 70)

IDENTIFICANDO COMPONENTES CONECTADOS DEL GRAFO MAPPER

Estructura de graph["links"]:
  Tipo: <class 'collections.defaultdict'>
  Número de elementos: 491
  Primer elemento: []
  Tipo del primer elemento: <class 'list'>

Procesando links...
  ⚠ Formato de link no reconocido: cube1_cluster0 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube1_cluster1 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube1_cluster2 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube1_cluster3 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster0 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster1 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster2 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube2_cluster3 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster0 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster1 (<class 'str'>)
  ⚠ Formato de link no reconocido: cube3_cluster2 (<class 'str'>)
  ⚠ Formato de link no reconocido:

## Análisis de Datos Topológicos (TDA) con KeplerMapper
### Segmentación de Mujeres Embarazadas según Exposición a Ácido Fólico y Riesgo Neonatal

### ¿Qué Hicimos?

1.  **Preparación de Datos:**
    *   Carga y limpieza del dataset de ingesta de Ácido Fólico (AF) y resultados neonatales.
    *   Ingeniería de nuevas características clave (ej., `AF_exposure`, `prematuro`, `riesgo_obstetrico`, `score_pan`).
    *   Imputación de valores nulos y escalamiento de características.

2.  **Configuración de KeplerMapper:**
    *   Definición de las *variables de interés* (`tda_features`) para el análisis topológico.
    *   Proyección de los datos a menor dimensión usando **t-SNE** (lente).
    *   Creación de una **cubierta (cover)** para segmentar el espacio de la lente (12 cubos, 45% solapamiento).
    *   Agrupamiento local dentro de cada segmento usando **DBSCAN**.

3.  **Generación del Grafo y Visualización:**
    *   Construcción de un **grafo topológico** con 61 nodos y 48 enlaces.
    *   Generación de una **visualización interactiva HTML** (`mapper_af_embarazo.html`), inicialmente coloreada por `AF_exposure`.